In [4]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import json
import re

In [13]:
# --------------------------
# 資料庫內各網紅影片數統計
# --------------------------

VIDEO_DIR = Path(
    "/home/ftp_246/data_5/多模態2026_ig_top200/"
    "Video_data_fullest/ig_vedios"
)

video_counts = []

for folder in VIDEO_DIR.iterdir():
    if not folder.is_dir():
        continue

    mp4_files = list(folder.rglob("*.mp4"))

    video_counts.append({
        "folder_name": folder.name,
        "video_count": len(mp4_files)
    })

df_video_counts = pd.DataFrame(video_counts)

df_video_counts = df_video_counts.sort_values(
    "video_count", ascending=False
)

# display(df_video_counts.head(20))

PROFILE_PATH = (
    "/home/yingtzudai/Multimodal_SPoD_Video/"
    "data/Influencer_Profile.csv"
)

df_profile = pd.read_csv(
    PROFILE_PATH,
    dtype={"ig_strong_id": str}
)

df_summary = df_profile.merge(
    df_video_counts,
    left_on="ig_id",
    right_on="folder_name",
    how="left",
    validate="one_to_one"
)

df_summary["video_count"] = (
    df_summary["video_count"].fillna(0).astype(int)
)

display(
    df_summary[
        ["ig_id", "AsiaKOL_category",
         "Followers", "Following", "video_count"]
    ].sort_values("video_count", ascending=False)
)

print("Total videos:", df_video_counts["video_count"].sum())



,ig_id,AsiaKOL_category,Followers,Following,video_count
139,dr.acne_scar_queen,"知識教育,醫療健康",189848,259,639
165,mayukichou,"時尚潮流,高階經理人",309258,1748,538
100,cocowine0205,"美食料理,趣味搞笑",157798,782,391
0,yga0721,趣味搞笑,1495461,501,344
14,ladyflavor_cui,NaN,369611,122,342
...,...,...,...,...,...
40,lynnwu0219_2,帶貨分潤,76276,93,0
117,alvin701,知識教育,187869,142,0
91,cheapaoe,"遊戲電玩,時事討論",24615,527,0
182,oldwangstock,理財創業,72829,10,0


Total videos: 19312


# Data Availability Check

In [15]:
BASE = Path("/home/ftp_246/data_5/多模態2026_ig_top200")

METADATA_DIR = BASE / "Metadata_by_Influencer"
VIDEO_DIR = BASE / "Video_data_fullest/ig_vedios"
JSON_DIR = BASE / "Prompt_Description/Fullset_v20"

# PROFILE_PATH = Path(
#     "/home/yingtzudai/Multimodal_SPoD_Video/data/Influencer_Profile.csv"
# )

# df_profile = pd.read_csv(
#     PROFILE_PATH,
#     dtype={"ig_id": str, "ig_strong_id": str}
# )



# 1. Metadata：讀取各網紅 CSV
metadata_rows = []

for csv_path in METADATA_DIR.rglob("*.csv"):
    df = pd.read_csv(
        csv_path,
        dtype={
            "id": str,
            "media_id": str,
            "short_code": str,
            "post_owner.username": str
        },
        low_memory=False
    )

    if "id" not in df.columns:
        print("Skipped (no id):", csv_path)
        continue

    for _, row in df.iterrows():
        metadata_rows.append({
            "ig_id": str(row.get("post_owner.username", "")).strip(),
            "post_id": str(row["id"]).strip(),
            "media_id": str(row.get("media_id", "")).strip(),
            "short_code": str(row.get("short_code", "")).strip(),
            "download_tool": str(row.get("DownloadTool", "")).strip(),
            "metadata_path": str(csv_path)
        })

df_metadata = pd.DataFrame(metadata_rows)

# 移除無效 ID 與重複貼文
df_metadata = df_metadata[
    df_metadata["post_id"].notna()
    & ~df_metadata["post_id"].isin(["", "nan", "None"])
].drop_duplicates(["ig_id", "post_id"])


# 2. MP4：列出影片
df_mp4 = pd.DataFrame([
    {
        "ig_id": path.parent.name,
        "video_filename": path.name,
        "video_path": str(path)
    }
    for path in VIDEO_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() == ".mp4"
])


# 3. JSON：列出 JSON 並讀取 videoName
json_rows = []

for path in JSON_DIR.rglob("*.json"):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        json_rows.append({
            "ig_id": path.parent.name,
            "json_filename": path.name,
            "video_filename": Path(
                str(data.get("videoName", ""))
            ).name,
            "json_path": str(path)
        })

    except (OSError, ValueError, TypeError) as e:
        print(f"JSON error: {path} | {e}")

df_json = pd.DataFrame(json_rows)


# 4. 初步統計
print("Metadata posts:", len(df_metadata))
print("MP4 files:", len(df_mp4))
print("JSON files:", len(df_json))

# display(df_metadata.head(5))
# display(df_mp4.head(5))
# display(df_json.head(5))

Metadata posts: 19443
MP4 files: 19312
JSON files: 19116


In [7]:
# 建立統計表
summary = df_profile[["ig_id"]].copy()

def add_counts(summary, df, column_name, unique_col=None):
    if unique_col:
        counts = df.groupby("ig_id")[unique_col].nunique()
    else:
        counts = df.groupby("ig_id").size()

    return summary.merge(
        counts.rename(column_name),
        on="ig_id",
        how="left"
    )

summary = add_counts(
    summary, df_metadata,
    "metadata_post_count", "post_id"
)

summary = add_counts(
    summary, df_mp4,
    "mp4_count"
)

summary = add_counts(
    summary, df_json,
    "json_count"
)


# JSON 與 MP4：以 ig_id + 完整影片檔名匹配
video_json = df_mp4.merge(
    df_json,
    on=["ig_id", "video_filename"],
    how="inner"
)

# 若同名檔案有多份，先去除重複對應
video_json = video_json.drop_duplicates(
    ["ig_id", "video_path"]
)

json_mp4_counts = (
    video_json.groupby("ig_id")["video_path"]
    .nunique()
    .rename("json_mp4_matched_count")
)

summary = summary.merge(
    json_mp4_counts,
    on="ig_id",
    how="left"
)


# Metadata 與 MP4：
# 僅接受檔名中有完整、唯一識別碼的情況
def match_post(video_name, posts):
    candidates = []

    for _, post in posts.iterrows():
        identifiers = [
            post["post_id"],
            post["media_id"],
            post["short_code"]
        ]

        for identifier in identifiers:
            if identifier in ("", "nan", "None"):
                continue

            # 避免短數字意外匹配其他 ID 的一部分
            pattern = (
                r"(?<![A-Za-z0-9])"
                + re.escape(identifier)
                + r"(?![A-Za-z0-9])"
            )

            if re.search(pattern, video_name):
                candidates.append(post["post_id"])
                break

    candidates = set(candidates)

    return next(iter(candidates)) if len(candidates) == 1 else None


matched_rows = []

for ig_id, videos in df_mp4.groupby("ig_id"):
    posts = df_metadata[df_metadata["ig_id"] == ig_id]

    for _, video in videos.iterrows():
        matched_rows.append({
            "ig_id": ig_id,
            "video_path": video["video_path"],
            "post_id": match_post(
                video["video_filename"], posts
            )
        })

df_matches = pd.DataFrame(
    matched_rows,
    columns=["ig_id", "video_path", "post_id"]
)

# 同一個 post_id 若對應多個影片，先視為不明確
valid = df_matches.dropna(subset=["post_id"]).copy()

unique_posts = (
    valid.groupby(["ig_id", "post_id"])["video_path"]
    .transform("nunique") == 1
)

valid = valid[unique_posts]

matched_counts = (
    valid.groupby("ig_id")["post_id"]
    .nunique()
    .rename("matched_video_count")
)

summary = summary.merge(
    matched_counts, on="ig_id", how="left"
)


# Metadata + MP4 + JSON 三者成功匹配
fully_matched = valid.merge(
    video_json[["ig_id", "video_path"]].drop_duplicates(),
    on=["ig_id", "video_path"],
    how="inner"
)

full_counts = (
    fully_matched.groupby("ig_id")["post_id"]
    .nunique()
    .rename("fully_matched_count")
)

summary = summary.merge(
    full_counts, on="ig_id", how="left"
)

count_cols = [
    "metadata_post_count",
    "mp4_count",
    "json_count",
    "json_mp4_matched_count",
    "matched_video_count",
    "fully_matched_count"
]

summary[count_cols] = (
    summary[count_cols].fillna(0).astype(int)
)

df_summary = summary.sort_values(
    "mp4_count", ascending=False
).reset_index(drop=True)

display(df_summary)

print("\nTotal:")
print(df_summary[count_cols].sum())

,ig_id,metadata_post_count,mp4_count,json_count,json_mp4_matched_count,matched_video_count,fully_matched_count
0,dr.acne_scar_queen,639,639,639,639,639,639
1,mayukichou,538,538,538,538,538,538
2,cocowine0205,391,391,391,391,391,391
3,yga0721,344,344,342,342,344,342
4,ladyflavor_cui,342,342,342,341,342,341
...,...,...,...,...,...,...,...
198,lynnwu0219_2,0,0,0,0,0,0
199,alvin701,1,0,0,0,0,0
200,cheapaoe,0,0,0,0,0,0
201,oldwangstock,0,0,0,0,0,0



Total:
metadata_post_count       19443
mp4_count                 19312
json_count                19116
json_mp4_matched_count    19104
matched_video_count       19310
fully_matched_count       19102
dtype: int64


In [17]:
# 看看有哪些影片沒有匹配到 Metadata
unmatched = df_matches[
    df_matches["post_id"].isna()
]

display(unmatched.head(20))
print(list(unmatched['video_path']))

# 查看 JSON 和 MP4 的成功匹配比例
print(
    "JSON-MP4 match rate:",
    round(
        len(video_json) / len(df_mp4) * 100, 2
    ) if len(df_mp4) else 0,
    "%"
)

,ig_id,video_path,post_id
1603,ala780705,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,None
13856,oldwangwang,/home/ftp_246/data_5/多模態2026_ig_top200/Video_d...,None


['/home/ftp_246/data_5/多模態2026_ig_top200/Video_data_fullest/ig_vedios/ala780705/ala780705-20250101143338-1258671062799200.mp4', '/home/ftp_246/data_5/多模態2026_ig_top200/Video_data_fullest/ig_vedios/oldwangwang/oldwangwang-20250327185143-1491235432413620.mp4']
JSON-MP4 match rate: 98.92 %


In [ ]:
# -----------------------------------
# 確認資料完整性
# -----------------------------------

totals = df_summary[
    [
        "metadata_post_count",
        "mp4_count",
        "json_count",
        "matched_video_count",
        "fully_matched_count"
    ]
].sum()

display(totals.to_frame("Total"))

print(
    "Overall full match rate:",
    f"{totals['fully_matched_count'] / totals['mp4_count']:.2%}"
    if totals["mp4_count"] > 0 else "N/A"
)

print(
    "Influencers with MP4:",
    (df_summary["mp4_count"] > 0).sum()
)

print(
    "Influencers with complete data:",
    (df_summary["fully_matched_count"] > 0).sum()
)

# ---------------------------------
# 找出資料不完整的網紅
# ---------------------------------
df_incomplete = df_summary[
    (df_summary["metadata_post_count"] != df_summary["mp4_count"])
    | (df_summary["mp4_count"] != df_summary["json_count"])
    | (df_summary["fully_matched_count"] != df_summary["mp4_count"])
].copy()

display(df_incomplete)

# ---------------------------------
# 檢查無法匹配的影片
# ---------------------------------
for _, row in unmatched.iterrows():
    print("\nInfluencer:", row["ig_id"])
    print("Video:", Path(row["video_path"]).name)

    related = df_metadata[
        df_metadata["ig_id"] == row["ig_id"]
    ]

    display(
        related[
            ["post_id", "media_id", "short_code"]
        ].head(10)
    )

,Total
metadata_post_count,19443
mp4_count,19312
json_count,19116
matched_video_count,19310
fully_matched_count,19102


Overall full match rate: 98.91%
Influencers with MP4: 197
Influencers with complete data: 196


,ig_id,metadata_post_count,mp4_count,json_count,json_mp4_matched_count,matched_video_count,fully_matched_count
3,yga0721,344,344,342,342,344,342
4,ladyflavor_cui,342,342,342,341,342,341
6,ethan_kuan_kuan,312,312,306,306,312,306
11,teacherchiang,287,285,277,277,285,277
12,aottergirls,284,283,283,283,283,283
...,...,...,...,...,...,...,...
159,byleway,29,29,18,18,29,18
167,children0704,24,23,21,21,23,21
196,oldwangwang,1,1,1,1,0,0
199,alvin701,1,0,0,0,0,0



Influencer: ala780705
Video: ala780705-20250101143338-1258671062799200.mp4


,post_id,media_id,short_code
3598,1258671062799202,1258671062799202,nan
3599,877386461572123,877386461572123,DESTdFeyZ4A
3600,1496410732045962,1496410732045962,DEUjDxwye9J
3601,1549701606086478,1549701606086478,DEcjBMeSgk6
3602,1226021142995469,1226021142995469,DExDDICSZfP
3603,939132858605246,939132858605246,DEzEOEdy-wC
3604,1242706491288958,1242706491288958,DEzdotSSsT6
3605,1425219295928182,1425219295928182,nan
3606,842177598824208,842177598824208,DE11KrgyJ0m
3607,742516408637400,742516408637400,DE4n210yy2l



Influencer: oldwangwang
Video: oldwangwang-20250327185143-1491235432413620.mp4


,post_id,media_id,short_code
15923,1491235432413628,1491235432413628,nan


In [ ]:
# ------------------------------
# 確認資料是否有重複
# ------------------------------
print(
    "Duplicated video paths:",
    df_mp4["video_path"].duplicated().sum()
)

print(
    "Duplicated JSON video names:",
    df_json.duplicated(
        ["ig_id", "video_filename"]
    ).sum()
)

print(
    "Duplicated Metadata post IDs:",
    df_metadata.duplicated(
        ["ig_id", "post_id"]
    ).sum()
)

Duplicated video paths: 0
Duplicated JSON video names: 0
Duplicated Metadata post IDs: 0
